# Section 7: Moran's I Immune Analysis

## Purpose
Quantify local spatial autocorrelation of immune-cell densities and their bivariate relationships with viral expression.


In [ ]:
%load_ext autoreload
%autoreload 2

## Setup


In [ ]:
  
# Imports: load Scanpy/GeoPandas/PySAL tooling for local spatial autocorrelation analysis
import scanpy as sc

## Data Loading and Marker Setup


In [ ]:
# Define plotting function to overlay cell features with vessel boundaries by core
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import pickle
from shapely.ops import unary_union

import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon
from shapely.geometry import Polygon
from shapely.ops import unary_union

def convert2gpd(df):
    # Step 1: Group the DataFrame by 'cell_id' or similar, assuming each cell has a unique ID
    # Ensure that your DataFrame has an identifier for each cell
    grouped = df.groupby('cell_id')

    # Step 2: Create polygons for each group of cell boundaries
    polygons = []

    for cell_id, group in grouped:
        # Extract the x and y coordinates for this cell
        points = group[['vertex_x', 'vertex_y']].values
        
        # Create a Polygon from these points
        # Ensure the points form a valid polygon (e.g., no crossing lines)
        if len(points) > 2:  # A polygon needs at least 3 points
            poly = Polygon(points)
            polygons.append({'cell_id': cell_id, 'geometry': poly})

    # Step 3: Create a GeoPandas DataFrame from the list of polygons
    gdf = gpd.GeoDataFrame(polygons)
    return gdf

def plot_spatial_feature_and_vessels(
    adata, 
    path_block_core, 
    feature=None, 
    feature_color_palette=None, 
    gene=None, 
    cmap=None,
    vessel_cmap='BuPu',
    linewidth=0.5,
    edgecolor='k',
    show=True, 
    na_color="#d3d3d3",
    save_path="../figures/cell_boundaries/",
    ax=None,
    vmin=None,
    vmax=None
):
    """
    Plots spatial data from an AnnData object for a given `path_block_core`, 
    optionally overlaying and outlining vessels.

    Parameters
    ----------
    adata : AnnData
        AnnData object containing spatial (e.g., .obs for metadata) and 
        expression (e.g., .X or .var_names) data.
    path_block_core : str
        The specific path block core to filter the data (must match a value in `adata.obs['path_block_core']`).
    feature : str, optional
        Column name in `adata.obs` to visualize. It can be categorical (in which 
        case `feature_color_palette` should be provided) or numerical.
    feature_color_palette : dict, optional
        Dictionary mapping unique categorical values to colors, e.g. 
        {"typeA": "#ff0000", "typeB": "#00ff00", ...}.
        Used only if `feature` is categorical.
    gene : str, optional
        Gene name to visualize from `adata.X`. If provided, the function plots
        the expression of this gene across the selected cells.
    cmap : str or Colormap, optional
        Colormap for numerical features or gene expression (default: "viridis").
    vessel_cmap : str or Colormap, optional
        Colormap used for vessels (default: "BuPu").
    linewidth : float, optional
        Line width for outlining cell boundaries (default: 0.5).
    edgecolor : str, optional
        Edge color for cell boundaries (default: 'k').
    show : bool, optional
        Whether to display the plot (default: True). If False, returns the Axes 
        object for further manipulation.
    na_color : str, optional
        Color to use for missing data (default: "#d3d3d3").
    save_path : str, optional
        Directory where the output figures are saved (default: "../figures/cell_boundaries/").
    ax : matplotlib.axes.Axes, optional
        Axes object on which to plot. If None, creates a new figure.
    vmin : float, optional
        Minimum value for the color scale when plotting numerical data. 
        If None, it defaults to the data minimum.
    vmax : float, optional
        Maximum value for the color scale when plotting numerical data. 
        If None, it defaults to the data maximum.

    Returns
    -------
    None or matplotlib.axes.Axes
        If `show` is False, returns the Axes object. Otherwise, displays the 
        plot and returns None.

    Notes
    -----
    - If both `feature` and `gene` are provided, the function will prompt the user
      to choose one, returning without creating a plot.
    - Ensure that `convert2gpd` is implemented or imported to correctly convert 
      the DataFrame of cell boundaries into a GeoDataFrame.
    - This function also attempts to load vessel outlines from a pickle file 
      ("../data/vessels.pkl") and overlay them on the plot.
    """
    # Ensure only one of feature or gene is used
    if feature and gene:
        print(f"Both `feature` ('{feature}') and `gene` ('{gene}') were provided. Please choose one.")
        return

    # Subset anndata for the selected path_block_core
    core_adata = adata[adata.obs['path_block_core'] == path_block_core]
    if core_adata.n_obs == 0:
        print(f"No cells found for path_block_core: {path_block_core}")
        return

    stage = core_adata.obs['Stage'].unique()[0]

    # Load vessels
    vessels_path = '../data/vessels.pkl'
    vessels = pickle.load(open(vessels_path, 'rb'))
    vessels_subset = vessels[vessels.path_block_core == path_block_core]

    # Process vessel geometries
    vessels_subset['geometry'] = vessels_subset['geometry'].apply(
        lambda geom: unary_union([g.buffer(5) for g in geom.geoms]).buffer(-5)
        if geom.geom_type in ['MultiPolygon', 'GeometryCollection'] else geom.buffer(5).buffer(-5)
    ).copy()
    
    vessels_subset['geometry'] = vessels_subset['geometry'].apply(
        lambda geom: unary_union([g.simplify(1) for g in geom.geoms])
        if geom.geom_type in ['MultiPolygon', 'GeometryCollection'] else geom.simplify(1)
    ).copy()

    # Load cell boundaries
    sample = core_adata.obs['sample_id'].unique()[0]
    cell_boundaries_gzip_path = f"../data/cell_boundaries/{sample}/cell_boundaries.csv.gz"
    cell_boundaries_df = pd.read_csv(cell_boundaries_gzip_path, compression='gzip')
    cell_boundaries_core_df = cell_boundaries_df[cell_boundaries_df['cell_id'].isin(core_adata.obs['cell_id'])]

    # Convert to GeoDataFrame
    gdf = convert2gpd(cell_boundaries_core_df)

    # Determine what to plot
    is_categorical = False  # Flag to check if feature is categorical

    if gene:
        # Check if gene exists
        if gene not in adata.var_names:
            print(f"Warning: Gene '{gene}' not found in anndata object.")
            return
        
        # Extract gene expression
        gene_expression = core_adata[:, gene].X.toarray().flatten()  # Ensure it's a NumPy array
        plot_column = "expression"
        data_label = f"{gene} Expression"
        gdf[plot_column] = gene_expression

    elif feature:
        # Check if feature exists
        if feature not in adata.obs.columns:
            print(f"Warning: Feature '{feature}' not found in adata.obs.")
            return
        
        plot_column = feature
        data_label = feature
        gdf = gdf.merge(core_adata.obs[['cell_id', feature]], on='cell_id', how='left')

        # Check if feature is categorical or numerical
        if feature_color_palette:  # If a color palette is provided, assume categorical
            is_categorical = True
            gdf[plot_column] = gdf[plot_column].astype(str)
            gdf['color'] = gdf[plot_column].map(feature_color_palette).fillna(na_color)  # Default to light gray for missing
    else:
        print("Please provide either a `feature` (metadata column) or a `gene` for visualization.")
        return

    # If feature is numerical (not categorical), apply colormap
    if not is_categorical:
        # Set colormap
        cmap = cmap if cmap else cm.viridis  # Default to "viridis"
        
        # Determine vmin, vmax if not provided
        data_vmin = gdf[plot_column].min()
        data_vmax = gdf[plot_column].max()
        if data_vmin == data_vmax:
            data_vmin, data_vmax = data_vmin - 0.1, data_vmax + 0.1  # Avoid colormap errors for constant values

        # Use passed-in vmin, vmax if provided, else use data-based
        vmin = vmin if (vmin is not None) else data_vmin
        vmax = vmax if (vmax is not None) else data_vmax
        
        norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
        gdf['color'] = gdf[plot_column].apply(lambda x: mcolors.to_hex(cmap(norm(x))) if pd.notnull(x) else na_color)

    # Plot
    if ax is None:
        fig, ax = plt.subplots(figsize=(20, 20), dpi=300)
    # else, reuse the existing ax

    # Plot the cell boundaries
    gdf.plot(color=gdf['color'], alpha=1, ax=ax, linewidth=linewidth, edgecolor=edgecolor)

    # Plot vessels (using their vessel_area for color mapping if numerical)
    # Here we assume the user may want to color them. Adjust as needed.
    if not vessels_subset.empty:
        # Create a colormap for vessels if you want them scaled by area
        if 'vessel_area' in vessels_subset.columns:
            vessel_min = vessels_subset['vessel_area'].min()
            vessel_max = vessels_subset['vessel_area'].max()
            vessel_norm = mcolors.Normalize(vmin=vessel_min, vmax=vessel_max)
            vessels_subset['vessel_color'] = vessels_subset['vessel_area'].apply(
                lambda x: mcolors.to_hex(plt.get_cmap(vessel_cmap)(vessel_norm(x)))
            )
        else:
            # If there's no 'vessel_area', just choose a single color
            vessels_subset['vessel_color'] = '#aaaaaa'  # fallback color

        lw = 4
        vessels_subset.plot(
            ax=ax,
            facecolor='none',
            edgecolor='white',
            linewidth=lw,
        )
        lw = 1.5
        vessels_subset.plot(
            ax=ax,
            facecolor='none',
            edgecolor='k',
            linewidth=lw,
        )

    # Invert y-axis and remove axis
    ax.invert_yaxis()
    ax.axis('off')

    # Create colorbar if feature/gene is numerical
    if not is_categorical:
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax, fraction=0.02, pad=0.04)
        cbar.set_label(data_label, fontsize=16)

    # Save figure if desired
    if save_path:
        save_dir = os.path.join(save_path, 'cell_boundaries', stage, data_label.replace(' ', '_'))
        os.makedirs(save_dir, exist_ok=True)
        save_filename = f"{save_dir}/cell_boundaries_{path_block_core}_{stage}_{data_label.replace(' ', '_')}_vessels.png"
        plt.savefig(save_filename, dpi=300, bbox_inches='tight', transparent=False)
        # print(f"Plot saved: {save_filename}")

    if show:
        plt.show()
    else:
        return ax


In [ ]:
# from splot.esda import lisa_cluster
import numpy as np
import pandas as pd
import geopandas as gpd
import esda
import libpysal
import matplotlib.pyplot as plt
from splot.esda import plot_local_autocorrelation


from splot._viz_utils import mask_local_auto, moran_hot_cold_spots, splot_colors
from matplotlib import colors


from esda.moran import Moran_Local
from esda.moran import Moran_Local_BV

color_dict_lisa = {
    'HH': '#ce0d00',
    'HL': '#ff8b83',
    'LL': '#005493',
    'LH': '#9194ff',
    'ns': 'lightgray'
}


def lisa_cluster(
    moran_loc, gdf, p=0.05, ax=None, legend=True, legend_kwds=None, two_category=True, **kwargs
):
    """
    Create a LISA Cluster map

    Parameters
    ----------
    moran_loc : esda.moran.Moran_Local or Moran_Local_BV instance
        Values of Moran's Local Autocorrelation Statistic
    gdf : geopandas dataframe instance
        The Dataframe containing information to plot. Note that `gdf` will be
        modified, so calling functions should use a copy of the user
        provided `gdf`. (either using gdf.assign() or gdf.copy())
    p : float, optional
        The p-value threshold for significance. Points will
        be colored by significance.
    ax : matplotlib Axes instance, optional
        Axes in which to plot the figure in multiple Axes layout.
        Default = None
    legend : boolean, optional
        If True, legend for maps will be depicted. Default = True
    legend_kwds : dict, optional
        Dictionary to control legend formatting options. Example:
        ``legend_kwds={'loc': 'upper left', 'bbox_to_anchor': (0.92, 1.05)}``
        Default = None
    **kwargs : keyword arguments, optional
        Keywords designing and passed to geopandas.GeoDataFrame.plot().

    Returns
    -------
    fig : matplotlip Figure instance
        Figure of LISA cluster map
    ax : matplotlib Axes instance
        Axes in which the figure is plotted

    Examples
    --------
    Imports

    >>> import matplotlib.pyplot as plt
    >>> from libpysal.weights.contiguity import Queen
    >>> from libpysal import examples
    >>> import geopandas as gpd
    >>> from esda.moran import Moran_Local
    >>> from splot.esda import lisa_cluster

    Data preparation and statistical analysis

    >>> guerry = examples.load_example('Guerry')
    >>> link_to_data = guerry.get_path('guerry.shp')
    >>> gdf = gpd.read_file(link_to_data)
    >>> y = gdf['Donatns'].values
    >>> w = Queen.from_dataframe(gdf)
    >>> w.transform = 'r'
    >>> moran_loc = Moran_Local(y, w)

    Plotting

    >>> fig = lisa_cluster(moran_loc, gdf)
    >>> plt.show()

    """
    # retrieve colors5 and labels from mask_local_auto
    _, colors5, _, labels = mask_local_auto(moran_loc, p=p)
    
    if two_category:
        
        replace_labels = {'ns':'ns',
                 'LL':'LL',
                  'LH':'ns',
                 'HL':'ns',
                  'HH':'HH'
                 }
        labels = [replace_labels[each] for each in labels]
        colors5.pop(1)
        
    # define ListedColormap
    hmap = colors.ListedColormap(colors5)

    if ax is None:
        figsize = kwargs.pop("figsize", None)
        fig, ax = plt.subplots(1, figsize=figsize)
    else:
        fig = ax.get_figure()

    # check for Polygon, else no edgecolor
    if gdf.geom_type.isin(["Polygon", "MultiPolygon"]).any():
        gdf.assign(cl=labels).plot(
            column="cl",
            categorical=True,
            k=2,
            cmap=hmap,
            linewidth=0.1,
            ax=ax,
            edgecolor="white",
            legend=legend,
            legend_kwds=legend_kwds,
            **kwargs
        )
    else:
        gdf.assign(cl=labels).plot(
            column="cl",
            categorical=True,
            k=2,
            cmap=hmap,
            linewidth=1.5,
            ax=ax,
            legend=legend,
            legend_kwds=legend_kwds,
            **kwargs
        )
    ax.set_axis_off()
    ax.set_aspect("equal")
    return fig, ax, labels

In [ ]:
# Load preprocessed AnnData and marker dictionary used for immune-spatial analysis
adata = sc.read_h5ad("../data/KS_adata_preprocessed.h5ad")

In [ ]:
# Process samples
samples = ['KS_TMA_1_0026870', 'KS_TMA_2_0026882', 'KS_TMA_3_0027198', 'KS_TMA_4_0026764', 'KS_TMA_5_0026776',
           'KS_TMA_6_0027092', 'KS_TMA_7_0027079', 'KS_TMA_8_0027273', 'KS_TMA_9_0026831', 'KS_TMA_10_0026828',
           'KS_TMA_11_0026930', 'KS_TMA_12_0026888', 'KS_TMA_13_0027077', 'KS_TMA_14_0027019', 'KS_TMA_15_0033811',
          'KS_TMA_16_0033809']

marker_list_df_all = pd.read_csv('../data/marker_list_dev_standardized_short.csv')
marker_list_df = marker_list_df_all.copy()
#marker_list_df = marker_list_df.query(f"Annotation not in ['Vascular Endothelial Cells', 'Lymphatic Endothelial Cells']")

# Define cell types and clusters
cell_types = list(marker_list_df.keys())

marker_list = marker_list_df
marker_list = marker_list.groupby('grouped_cts')['Gene'].unique().reset_index()
marker_list = marker_list.set_index('grouped_cts')['Gene'].apply(list).to_dict()

complete_cell_types = list(marker_list_df['grouped_cts'].unique())
marker_genes_dict = marker_list

In [ ]:
# Build endothelial marker panels from the marker table for context plotting
EC_markers = {"Lymphatic Endothelial Cells": marker_list_df[marker_list_df['Annotation'] == "Lymphatic Endothelial Cells"]['Gene'].tolist(),
 "Vascular Endothelial Cells": marker_list_df[marker_list_df['Annotation'] == "Vascular Endothelial Cells"]['Gene'].tolist(),
 "Endothelial Cells": marker_list_df[marker_list_df['Annotation'] == "Endothelial Cells"]['Gene'].tolist(),}

In [ ]:
# Import custom scatter plotting helper used for spatial overlays
from utils import scatter_df

In [ ]:
# Subset macrophages for focused immune spatial inspection
adata_mph = adata[adata.obs.broad_cell_types == 'Macrophages']

In [ ]:
# Extract one representative core for all-cell exploratory visualization
df_all = adata[adata.obs.path_block_core == 'SYD06-2641_2D_1'].obs

In [ ]:
# Extract matching macrophage-only view for the same core
df = adata_mph[adata_mph.obs.path_block_core == 'SYD06-2641_2D_1'].obs
df

In [ ]:

# Define color mapping for broad cell types used in exploratory spatial plots
broad_cell_types_color_mapping = {\
    'Lymphatic Endothelial Cells': '#ffb695',
    'Macrophages': '#ff40ff',
    'Vascular Endothelial Cells': '#a4e000',
    'Pericytes': '#9f7704',
    'Fibroblasts': '#c7d0c0',
    'T-cells': '#941100',
    'Keratinocytes': '#181c82',
    'Dendritic cells': '#ff9300',
    'Spinous to Granular Cells': '#034cff',
    'Pilosebaceous Cells': '#bbbde2',
    'B-cells': '#f12d00',
    'Melanocytes': '#00bbbf'
}

niche_colors = {
    #SKIN
    "Basal Dermis": "#b299e3",
    "Differentiated Epidermis": "#ffd000",
 
    # STROMA
    "Stroma": "#646500",
    "TA VEC Stroma": "#00e50c",
    "VEC Stroma": "#cccc33",
    
    # IMMUNE
    "Macrophage Immune Stroma": "#00dbf4",
    "T-cell Immune Stroma": "#0051f9",
    "Immune": "#c100f9",
 
    # TUMOR
    "Tumor Core": "#450000",
    "Tumor": "#eb0000",
    "Tumor Boundary": "#faa0aa"
}

## Local Moran's I Workflow


In [ ]:
# Import Moran's I and local autocorrelation utilities for immune density analysis
import numpy as np
import geopandas as gpd
import libpysal
import esda
import matplotlib.pyplot as plt
import os
from concurrent.futures import ProcessPoolExecutor, as_completed

# Define your immune cell types
n_immune_celltypes = ['n_B-cells', 'n_Dendritic cells', 'n_KSHV+ Macrophages', 
                      'n_KSHV+ Dendritic cells', 'n_KSHV+ T-cells', 
                      'n_Macrophages', 'n_T-cells']

# Function to perform LISA analysis for a given adata subset
def perform_lisa_analysis(adata_subset, path_block, adata):
    # Create a GeoDataFrame from the DataFrame using x and y columns
    geometry = gpd.points_from_xy(adata_subset.obs['local_x'], adata_subset.obs['local_y'])
    gdf = gpd.GeoDataFrame(adata_subset.obs, geometry=geometry)

    # Calculate queen contiguity weights
    w = libpysal.weights.Queen.from_dataframe(gdf)

    # Loop through each immune cell type and perform LISA analysis
    for ctype in n_immune_celltypes:
        feat = np.array(adata_subset.obs[ctype])
        lisa = esda.Moran_Local(feat, w)

        fig, ax, labels = lisa_cluster(
            lisa,
            gdf,
            p=0.05,
            legend=True,
            legend_kwds={"loc": "upper left", "bbox_to_anchor": (0.92, 1.05)},
            s=1,
            figsize=(17, 17),
            two_category=False
        )

        # Save the labels in the adata_subset
        adata_subset.obs[f'lisa_{ctype}'] = labels

        # Pooling results back into the original adata
        adata.obs.loc[adata_subset.obs.index, f'lisa_{ctype}'] = labels

        # Get the stage for output directory
        stage = adata_subset.obs['Stage'].unique()[0]

        output_directory = f'../figures/figure_6/local_morans/{stage}/'
        os.makedirs(output_directory, exist_ok=True)

        # Save the figure
        plt.savefig(f'{output_directory}local_morans_{path_block}_{ctype}.png', dpi=350)
        plt.close(fig)  # Close the figure to avoid display

    print(f'Completed LISA analysis for path_block: {path_block}')

# Get unique path_block_core identifiers
unique_path_blocks = adata.obs['path_block_core'].unique()

# Use ProcessPoolExecutor to parallelize the LISA analysis for each unique path_block_core
with ProcessPoolExecutor() as executor:
    futures = {
        executor.submit(perform_lisa_analysis, adata[adata.obs.path_block_core == path_block], path_block, adata): path_block
        for path_block in unique_path_blocks
    }

    for future in as_completed(futures):
        path_block = futures[future]
        try:
            future.result()  # This will also raise any exceptions that occurred in the thread
        except Exception as e:
            print(f'Error occurred for path_block {path_block}: {e}')

In [ ]:
# Quick sanity check of the full AnnData object before per-core Moran analysis
adata

In [ ]:
# Define immune density features to evaluate with local Moran's I
n_immune_celltypes = ['n_B-cells', 'n_Dendritic cells', 'n_KSHV+ Macrophages', 'n_KSHV+ Dendritic cells',  'n_KSHV+ T-cells', 'n_Macrophages', 'n_T-cells',]


adata_one_core = adata[adata.obs.path_block_core == 'SYD19-2015_A_5']

stage = adata_one_core.obs['Stage'].unique()[0]

# Create a GeoDataFrame from the DataFrame using x and y columns
geometry = gpd.points_from_xy(adata_one_core.obs['local_x'], adata_one_core.obs['local_y'])
gdf = gpd.GeoDataFrame(adata_one_core.obs, geometry=geometry)

# Calculate queen contiguity weights
w = libpysal.weights.Queen.from_dataframe(gdf)

for ctype in n_immune_celltypes:
    # Convert the infection column to a numpy array
    feat = np.array(adata_one_core.obs[ctype])
    
    # Perform LISA analysis
    lisa = esda.Moran_Local(feat, w)
    
    fig, ax, labels = lisa_cluster(
        lisa,
        gdf,
        p=0.05,
        legend=True,
        legend_kwds={"loc": "upper left", "bbox_to_anchor": (0.92, 1.05)},
        s=1,
        figsize=(17,17),
        two_category=False
    )
    
    adata_one_core.obs[f'lisa_{ctype}'] = labels
    
    plt.savefig(f'../figures/figure_6/local_morans/{stage}/local_morans_IMMUNE_{ctype}.png', dpi=350)
    plt.show()



In [ ]:
# Select a single path-block core for detailed local autocorrelation analysis
adata_one_core = adata[adata.obs.path_block_core == 'SYD19-2015_A_5']
adata_one_core

In [ ]:
# Create a GeoDataFrame from the DataFrame using x and y columns
geometry = gpd.points_from_xy(adata_one_core.obs['local_x'], adata_one_core.obs['local_y'])
gdf = gpd.GeoDataFrame(adata_one_core.obs, geometry=geometry)

In [ ]:
# Calculate queen contiguity weights
w = libpysal.weights.Queen.from_dataframe(gdf)

In [ ]:
# Inspect one-core AnnData slice prior to LISA computation
adata_one_core

In [ ]:
# Reconfirm immune feature list for iterative per-feature LISA analysis loop
n_immune_celltypes = ['n_B-cells', 'n_Dendritic cells', 'n_KSHV+ Macrophages', 'n_KSHV+ Dendritic cells',  'n_KSHV+ T-cells', 'n_Macrophages', 'n_T-cells',]

In [ ]:
# Compute Local Moran's I per immune feature and generate LISA cluster maps
for ctype in n_immune_celltypes:
    # Convert the infection column to a numpy array
    feat = np.array(adata_one_core.obs[ctype])
    
    # Perform LISA analysis
    lisa = esda.Moran_Local(feat, w)
    
    fig, ax, labels = lisa_cluster(
        lisa,
        gdf,
        p=0.05,
        legend=True,
        legend_kwds={"loc": "upper left", "bbox_to_anchor": (0.92, 1.05)},
        s=1,
        figsize=(17,17),
        two_category=False
    )
    
    adata_one_core.obs[f'lisa_{ctype}'] = labels
    
    plt.savefig(f'../figures/figure_6/local_morans/local_morans_IMMUNE_{ctype}.png', dpi=350)
    plt.show()



In [ ]:
# Convert the infection column to a numpy array
feat = np.array(adata_one_core.obs['n_Macrophages'])

# Perform LISA analysis
lisa = esda.Moran_Local(feat, w)

fig, ax, labels = lisa_cluster(
    lisa,
    gdf,
    p=0.05,
    legend=True,
    legend_kwds={"loc": "upper left", "bbox_to_anchor": (0.92, 1.05)},
    s=1,
    figsize=(17,17),
    two_category=False
)

adata_one_core.obs['lisa_n_Macrophages'] = labels

# plt.savefig(f'results/morans_.png', dpi=350)
plt.show()

In [ ]:
# Import scatter helper for bivariate Moran visualization overlays
from utils import scatter_df

## Cross-Correlation (Bivariate Moran's I)


In [ ]:

# Convert the infection column to a numpy array
feat1 = np.array(adata_one_core.obs["n_Macrophages"])
feat2 = np.array(adata_one_core[:, 'KSHV.K2'].X.todense())

# Perform LISA analysis
lisa = esda.Moran_Local_BV(feat1, feat2, w)

_, _, _, labels = mask_local_auto(lisa, p=0.05)

In [ ]:
# Store bivariate LISA labels (KSHV.K2 vs macrophage density) in adata.obs
adata_one_core.obs['lisa_BV KSHV.K2 n_Macrophages'] = labels
adata_one_core.obs['lisa_BV KSHV.K2 n_Macrophages'] = adata_one_core.obs['lisa_BV KSHV.K2 n_Macrophages'].astype('category')

## Visualization and Differential Expression


In [ ]:
# Map LISA labels to colors for spatial rendering of bivariate autocorrelation classes
plot_colors_list = [color_dict_lisa[label] for label in labels]

scatter_df(adata_one_core.obs, 
           'local_x', 'local_y', 
           c=plot_colors_list, s=2
          )

In [ ]:
# Define final LISA color palette and render feature/vessel overlay map
color_dict_lisa = {
    'HH': '#ce0d00',
    'HL': '#ff8b83',
    'LL': '#005493',
    'LH': '#9194ff',
    'ns': 'w'}

color_dict_lisa = {
    'HH': '#ce0d00',
    'HL': 'w',
    'LL': '#005493',
    'LH': 'w',
    'ns': 'w'}

plot_spatial_feature_and_vessels(adata_one_core, 
                                 'SYD19-2015_A_5', 
                                 feature=f'lisa_BV KSHV.K2 n_Macrophages',
                                 gene=None, 
                                 linewidth=0.1,
                                 feature_color_palette = color_dict_lisa,
                                 # cmap=cm.Reds,
                                 # vmin=0,
                                 # vmax=1,
                                 na_color='w',
                                 save_path="../figures/")

In [ ]:
# Import proportion plotting helper for summarizing LISA class compositions
from plot_utils import proportion

In [ ]:
# Sanity check AnnData after LISA annotations and derived label columns
adata

In [ ]:
# Define LISA class palette for infected macrophage local autocorrelation summaries
color_dict_lisa = {
    'HH': '#ce0d00',
    'HL': '#ff8b83',
    'LL': '#005493',
    'LH': '#9194ff',
    'ns': 'lightgray'}

proportion(
    adata,
    group_key='niche_with_tumor_proximity',
    label_key=f'lisa_n_KSHV+ Macrophages',
    figsize=(5, 4), rotation_xlabel=90,
    dpi=200,
    palette=color_dict_lisa,
    save=f'../figures/figure_6/infected_macrophages_LISA_proportions.pdf'
)

In [ ]:
# Define alternate reduced LISA palette emphasizing HH/LL classes
color_dict_lisa = {
    'HH': '#ce0d00',
    'HL': '#ff8b83',
    'LL': '#005493',
    'LH': '#9194ff',
    'ns': 'lightgray'}

proportion(
    adata,
    group_key='niche_with_tumor_proximity',
    label_key=f'lisa_n_Macrophages',
    figsize=(5, 4), rotation_xlabel=90,
    dpi=200,
    palette=color_dict_lisa,
    save=f'../figures/figure_6/overall_macrophages_LISA_proportions.pdf'
)

In [ ]:
# Run differential expression between LISA classes to identify associated genes
sc.tl.rank_genes_groups(
    adata_one_core, groupby='lisa_BV KSHV.K2 n_Macrophages', #reference='LL'
)

In [ ]:
# Plot ranked DE genes for selected LISA class comparisons
sc.pl.rank_genes_groups(
    adata_one_core, groupby='lisa_BV KSHV.K2 n_Macrophages', #reference='LL', groups=['HH']
)

In [ ]:
# Visualize DE markers as a dot plot across LISA classes
sc.pl.rank_genes_groups_dotplot(
    adata_one_core, groupby='lisa_BV KSHV.K2 n_Macrophages',  n_genes=5, #groups = ['HH']
)

In [ ]:
# Visualize DE markers as a heatmap across LISA classes
sc.pl.rank_genes_groups_heatmap(
    adata_one_core, groupby='lisa_BV KSHV.K2 n_Macrophages', groups = ['HH'], 
)